In [1]:
import json
from os import listdir, makedirs
from pathlib import Path
from typing import TypedDict

import numpy as np
import torch
from PIL import Image
from sklearn.preprocessing import StandardScaler

from dinosaw.linear_probe import (
    RampTypes,
    Regressor,
    gen_sample_mask,
    get_model as get_reg_model,
    get_ramp,
)
from dinosaw.utils import get_features
from dinosaw.wrappers import ModelTypes, get_models

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [2]:
enabled_models: tuple[ModelTypes, ...] = (
    "dinov2_s",
    "alibi_coco_dinov2_s",
    # "dinov2_b",
    # "cb_dinov2_s",
    # "dvt_dinov2_s",
    # "alibi_dinov3_s+",
)

models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_init_weights', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_pos_embed', '_r

In [3]:
micro_folder = "data/linear_probe/homog_micros"
texture_folder = "data/linear_probe/texture_ds"
MAX_IMGS: int | None = 60  # Set to None to use all images in folder
IMG_SIZE: tuple[int, int] = (518, 518)  # Resize so train/test patch masks align across images A and B


def featurise_dataset_all_models(
    folder: str,
    models_dict: dict,
    max_imgs: int | None = MAX_IMGS,
    img_size: tuple[int, int] = IMG_SIZE,
) -> dict[ModelTypes, list[np.ndarray]]:
    image_files = sorted(
        f
        for f in listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
    )
    if max_imgs is not None:
        image_files = image_files[:max_imgs]

    ds_features: dict[ModelTypes, list[np.ndarray]] = {m: [] for m in models_dict.keys()}
    for img_file in image_files:
        img_path = f"{folder}/{img_file}"
        img = Image.open(img_path).convert("RGB").resize(img_size)
        for model_name, model in models_dict.items():
            feats = get_features(model, img, channel_last=True)
            ds_features[model_name].append(np.nan_to_num(feats, nan=0.0))
    return ds_features


micro_features = featurise_dataset_all_models(micro_folder, models)
texture_features = featurise_dataset_all_models(texture_folder, models)
n_micro_imgs = len(next(iter(micro_features.values())))
n_texture_imgs = len(next(iter(texture_features.values())))

In [4]:
class CrossImageProbeResult(TypedDict):
    a_train_r2: list[float]
    a_test_r2: list[float]
    b_train_r2: list[float]
    b_test_r2: list[float]
    a_preds: list[np.ndarray]
    b_preds: list[np.ndarray]
    per_channel_a_train: list[np.ndarray] | None
    per_channel_a_test: list[np.ndarray] | None
    per_channel_b_train: list[np.ndarray] | None
    per_channel_b_test: list[np.ndarray] | None


def _eval_per_channel_r2(
    Z: np.ndarray,
    y_true: np.ndarray,
    w: np.ndarray,
    y_mean: np.ndarray,
) -> np.ndarray:
    pred = Z[:, :, None] * w[None, :, :] + y_mean[None, None, :]
    ss_res = ((y_true[:, None, :] - pred) ** 2).sum(axis=0)
    ss_tot = ((y_true - y_true.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)[None, :]
    return (1.0 - ss_res / ss_tot).mean(axis=-1)


def do_cross_image_linear_probe(
    src_dataset: list[np.ndarray],
    ramp_type: RampTypes,
    targ_dataset: list[np.ndarray] | None = None,
    mask_step: int = 6,
    mask_cutoff_frac: float = 1.0,
    random_mask: bool = True,
    probe_by_channel: bool = False,
    scale_feats: bool = True,
    regressor: Regressor = "ridge",
    all_to_all: bool = False,
) -> CrossImageProbeResult:
    """Train a linear probe once per image A in `src_dataset` and evaluate on A (train/test) and B in `targ_dataset` (train/test)."""
    same_dataset = targ_dataset is None or targ_dataset is src_dataset
    if targ_dataset is None:
        targ_dataset = src_dataset

    n_src_imgs = len(src_dataset)
    n_targ_imgs = len(targ_dataset)
    h, w, c = src_dataset[0].shape
    target = get_ramp(ramp_type, h, w)
    d = target.shape[-1]

    a_train_r2: list[float] = []
    a_test_r2: list[float] = []
    b_train_r2: list[float] = []
    b_test_r2: list[float] = []
    a_preds: list[np.ndarray] = []
    b_preds: list[np.ndarray] = []

    per_channel_a_train: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_a_test: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_b_train: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_b_test: list[np.ndarray] | None = [] if probe_by_channel else None

    for i in range(n_src_imgs):
        feats_A = src_dataset[i]
        sample_mask = gen_sample_mask(
            (h, w), ramp_type, step=mask_step, cutoff_frac=mask_cutoff_frac, random_mask=random_mask
        )

        X_A_train = feats_A[sample_mask, :]
        X_A_test = feats_A[~sample_mask, :]
        X_A_full = feats_A.reshape(-1, c)
        y_train = target[sample_mask, :]
        y_test = target[~sample_mask, :]

        if scale_feats:
            a_scaler = StandardScaler().fit(X_A_full)
            X_A_train_sc = a_scaler.transform(X_A_train)
            X_A_test_sc = a_scaler.transform(X_A_test)
            X_A_full_sc = a_scaler.transform(X_A_full)
        else:
            X_A_train_sc = X_A_train
            X_A_test_sc = X_A_test
            X_A_full_sc = X_A_full

        reg = get_reg_model(regressor)
        reg.fit(X_A_train_sc, y_train)

        a_train_r2.append(float(reg.score(X_A_train_sc, y_train)))
        a_test_r2.append(float(reg.score(X_A_test_sc, y_test)))
        a_preds.append(reg.predict(X_A_full_sc).reshape(h, w, d))

        if probe_by_channel:
            alpha = 1.0 if regressor == "ridge" else 0.0
            y_mean = y_train.mean(axis=0)
            denom = (X_A_train_sc ** 2).sum(axis=0) + alpha
            w_ch = np.einsum("nc,nd->cd", X_A_train_sc, y_train - y_mean[None, :]) / denom[:, None]
            assert per_channel_a_train is not None and per_channel_a_test is not None
            per_channel_a_train.append(_eval_per_channel_r2(X_A_train_sc, y_train, w_ch, y_mean))
            per_channel_a_test.append(_eval_per_channel_r2(X_A_test_sc, y_test, w_ch, y_mean))

        if all_to_all:
            b_indices = [j for j in range(n_targ_imgs) if not (same_dataset and j == i)]
        else:
            b_indices = [(i + 1) % n_targ_imgs]
        b_pred_accum = np.zeros((h, w, d), dtype=np.float64)

        for j in b_indices:
            feats_B = targ_dataset[j]
            X_B_train = feats_B[sample_mask, :]
            X_B_test = feats_B[~sample_mask, :]
            X_B_full = feats_B.reshape(-1, c)

            if scale_feats:
                b_scaler = StandardScaler().fit(X_B_full)
                X_B_train_sc = b_scaler.transform(X_B_train)
                X_B_test_sc = b_scaler.transform(X_B_test)
                X_B_full_sc = b_scaler.transform(X_B_full)
            else:
                X_B_train_sc = X_B_train
                X_B_test_sc = X_B_test
                X_B_full_sc = X_B_full

            b_train_r2.append(float(reg.score(X_B_train_sc, y_train)))
            b_test_r2.append(float(reg.score(X_B_test_sc, y_test)))
            b_pred_accum += reg.predict(X_B_full_sc).reshape(h, w, d) / len(b_indices)

            if probe_by_channel:
                assert per_channel_b_train is not None and per_channel_b_test is not None
                per_channel_b_train.append(_eval_per_channel_r2(X_B_train_sc, y_train, w_ch, y_mean))
                per_channel_b_test.append(_eval_per_channel_r2(X_B_test_sc, y_test, w_ch, y_mean))

        b_preds.append(b_pred_accum)

    return {
        "a_train_r2": a_train_r2,
        "a_test_r2": a_test_r2,
        "b_train_r2": b_train_r2,
        "b_test_r2": b_test_r2,
        "a_preds": a_preds,
        "b_preds": b_preds,
        "per_channel_a_train": per_channel_a_train,
        "per_channel_a_test": per_channel_a_test,
        "per_channel_b_train": per_channel_b_train,
        "per_channel_b_test": per_channel_b_test,
    }

In [5]:
class ScoreSummary(TypedDict):
    mean: float
    std: float
    all_scores: list[float]


RAMP: RampTypes = "lr+ud"
ALL_TO_ALL: bool = False
SCALE_FEATS: bool = True
MASK_CUTOFF_FRAC: float = 1.0
STEP: int = 6
RANDOM_MASK: bool = True

probe_tasks = (
    "A_to_A_micro",      # 1) Image A -> Image A on micrograph DS (standard test R^2)
    "A_to_B_micro",      # 2) Image A -> Image B on micrograph DS
    "A_to_B_texture",    # 3) Image A -> Image B on micrograph -> texture DS
)

# scores[task_key][model_name] -> ScoreSummary
scores: dict[str, dict[str, ScoreSummary]] = {t: {} for t in probe_tasks}

print(
    f"{'Model':<22} | "
    f"{'A -> A (micro)':<17} | "
    f"{'A -> B (micro)':<17} | "
    f"{'A -> B (texture)':<17}"
)
print("-" * 82)

for model_name in enabled_models:
    # 1 & 2: Micrograph A -> Micrograph A (standard) and Micrograph A -> Micrograph B
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    res_micro = do_cross_image_linear_probe(
        micro_features[model_name],
        RAMP,
        targ_dataset=micro_features[model_name],
        mask_step=STEP,
        mask_cutoff_frac=MASK_CUTOFF_FRAC,
        random_mask=RANDOM_MASK,
        probe_by_channel=False,
        scale_feats=SCALE_FEATS,
        regressor="ridge",
        all_to_all=ALL_TO_ALL,
    )

    # 3: Micrograph A -> Texture B
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    res_cross_ds = do_cross_image_linear_probe(
        micro_features[model_name],
        RAMP,
        targ_dataset=texture_features[model_name],
        mask_step=STEP,
        mask_cutoff_frac=MASK_CUTOFF_FRAC,
        random_mask=RANDOM_MASK,
        probe_by_channel=False,
        scale_feats=SCALE_FEATS,
        regressor="ridge",
        all_to_all=ALL_TO_ALL,
    )

    for task_key, raw_vals in (
        ("A_to_A_micro", res_micro["a_test_r2"]),
        ("A_to_B_micro", res_micro["b_test_r2"]),
        ("A_to_B_texture", res_cross_ds["b_test_r2"]),
    ):
        scores[task_key][model_name] = {
            "mean": float(np.mean(raw_vals)),
            "std": float(np.std(raw_vals)),
            "all_scores": [float(v) for v in raw_vals],
        }

    s_aa = scores["A_to_A_micro"][model_name]
    s_ab_m = scores["A_to_B_micro"][model_name]
    s_ab_t = scores["A_to_B_texture"][model_name]
    print(
        f"{model_name:<22} | "
        f"{s_aa['mean']:6.2f} ± {s_aa['std']:5.2f}   | "
        f"{s_ab_m['mean']:6.2f} ± {s_ab_m['std']:5.2f}   | "
        f"{s_ab_t['mean']:6.2f} ± {s_ab_t['std']:5.2f}"
    )

Model                  | A -> A (micro)    | A -> B (micro)    | A -> B (texture) 
----------------------------------------------------------------------------------
dinov2_s               |   0.80 ±  0.06   |   0.59 ±  0.11   |   0.49 ±  0.14
alibi_coco_dinov2_s    |   0.00 ±  0.41   |  -0.71 ±  0.67   |  -0.56 ±  0.56


In [6]:
class ModelMeta(TypedDict, total=False):
    name: str
    d: str
    pe: str
    supervised: bool
    bold: bool


DEFAULT_MODEL_METADATA: dict[str, ModelMeta] = {
    "dino_s": {"name": "DINO", "d": "S", "pe": "Learned", "supervised": False, "bold": False},
    "dino_b": {"name": "DINO", "d": "B", "pe": "Learned", "supervised": False, "bold": False},
    "dinov2_s": {"name": "DINOv2", "d": "S", "pe": "Learned", "supervised": False, "bold": False},
    "tr_dinov2_s": {"name": "DINOv2", "d": "S", "pe": "Learned", "supervised": False, "bold": False},
    "dinov2_b": {"name": "DINOv2", "d": "B", "pe": "Learned", "supervised": False, "bold": False},
    "dinov3_s+": {"name": "DINOv3", "d": "S+", "pe": "RoPE", "supervised": False, "bold": False},
    "dinov3_b": {"name": "DINOv3", "d": "B", "pe": "RoPE", "supervised": False, "bold": False},
    "mae_b": {"name": "ViT (MAE)", "d": "B", "pe": "Learned", "supervised": False, "bold": False},
    "sam_b": {"name": "SAM", "d": "256", "pe": "Learned", "supervised": True, "bold": False},
    "in1k_b": {"name": "ViT (IN1k)", "d": "B", "pe": "Learned", "supervised": True, "bold": False},
    "deit_s": {"name": "DEiT", "d": "B", "pe": "Learned", "supervised": True, "bold": False},
    "clip_b": {"name": "CLIP", "d": "B", "pe": "Learned", "supervised": True, "bold": False},
    "eva02_b": {"name": "EVA02", "d": "B", "pe": "RoPE", "supervised": True, "bold": False},
    "cb_dinov2_s": {"name": "DINOv2 (CB)", "d": "S", "pe": "Learned", "supervised": False, "bold": False},
    "dvt_dinov2_s": {"name": "DVT", "d": "S", "pe": "Learned", "supervised": False, "bold": False},
    "alibi_dinov2_s": {"name": "ALiBi-Dv2", "d": "S", "pe": "ALiBI", "supervised": False, "bold": True},
    "alibi_coco_dinov2_s": {"name": "ALiBi-Dv2", "d": "S", "pe": "ALiBI", "supervised": False, "bold": True},
    "alibi_dinov3_s+": {"name": "ALiBi-Dv3", "d": "S+", "pe": "ALiBI", "supervised": False, "bold": True},
    "nope_dinov2_s": {"name": "NoPE", "d": "S", "pe": "None", "supervised": False, "bold": False},
    "sinusoid_dinov2_s": {"name": "Sinusoid", "d": "S", "pe": "Sinusoid", "supervised": False, "bold": False},
    "sinusoid_dinov2_s_cb": {"name": "Sinusoid (CB)", "d": "S", "pe": "Sinusoid", "supervised": False, "bold": False},
}


def _format_text(text: str, bold: bool) -> str:
    return rf"\textbf{{{text}}}" if bold else text


def _format_score_cell(
    mean: float,
    std: float,
    *,
    red_threshold: float = 0.4,
    show_std: bool = False,
    bold: bool = False,
) -> str:
    color = "red!25" if mean > red_threshold else "green!25"
    val_str = f"{mean:.2f} $\\pm$ {std:.2f}" if show_std else f"{mean:.2f}"
    if bold:
        val_str = rf"\textbf{{{val_str}}}"
    return rf"\cellcolor{{{color}}} {val_str}"


def render_cross_probe_latex_table(
    scores_dict: dict[str, dict[str, ScoreSummary]],
    model_list: tuple[str, ...] | list[str],
    *,
    show_std: bool = False,
    show_supervised_col: bool = False,
    use_multirow: bool = True,
    red_threshold: float = 0.4,
    caption: str = (
        r"$(x, y)$ linear probe $R^2$ scores trained on micrograph images (A) and evaluated on: "
        r"held-out patches of the same image ($\text{A} \to \text{A}$, `micro'), "
        r"patches of another micrograph image ($\text{A} \to \text{B}$, `micro'), "
        r"and patches of a texture image ($\text{A} \to \text{B}$, `texture')."
    ),
    label: str = "tab:cross_image_linear_probe",
) -> str:
    task_columns = [
        ("A_to_A_micro", r"$R^2_{\text{A} \to \text{A (micro)}}$"),
        ("A_to_B_micro", r"$R^2_{\text{A} \to \text{B (micro)}}$"),
        ("A_to_B_texture", r"$R^2_{\text{A} \to \text{B (texture)}}$"),
    ]

    resolved_meta: dict[str, ModelMeta] = {}
    for m in model_list:
        resolved_meta[m] = dict(
            DEFAULT_MODEL_METADATA.get(
                m,
                {"name": m, "d": "", "pe": "", "supervised": False, "bold": False},
            )
        )  # type: ignore[assignment]

    # Pre-calculate multirow spans for consecutive models sharing the same display name
    name_spans: list[int] = [1] * len(model_list)
    if use_multirow:
        i = 0
        while i < len(model_list):
            j = i + 1
            while (
                j < len(model_list)
                and resolved_meta[model_list[j]].get("name") == resolved_meta[model_list[i]].get("name")
                and resolved_meta[model_list[j]].get("bold", False) == resolved_meta[model_list[i]].get("bold", False)
            ):
                j += 1
            span = j - i
            if span > 1:
                name_spans[i] = span
                for k in range(i + 1, j):
                    name_spans[k] = 0
            i = j

    n_meta_cols = 4 if show_supervised_col else 3
    col_spec = "l " + " ".join(["c"] * (n_meta_cols - 1 + len(task_columns)))

    header_cols = ["Model", r"\textit{d}", "PE"]
    if show_supervised_col:
        header_cols.append("Supervised")
    for _, col_header in task_columns:
        header_cols.append(col_header)
    header_line = "        " + " & ".join(header_cols) + r" \\"

    row_lines: list[str] = []
    for idx, m in enumerate(model_list):
        meta = resolved_meta[m]
        is_bold = bool(meta.get("bold", False))
        disp_name = _format_text(meta.get("name", m), is_bold)
        d_str = _format_text(meta.get("d", ""), is_bold)
        pe_str = _format_text(meta.get("pe", ""), is_bold)

        span = name_spans[idx]
        if span > 1:
            model_col = rf"\multirow{{{span}}}{{*}}{{{disp_name}}}"
        elif span == 1:
            model_col = disp_name
        else:
            model_col = ""

        row_cells = [model_col, d_str, pe_str]
        if show_supervised_col:
            sup_str = r"$\checkmark$" if meta.get("supervised", False) else r"$\times$"
            row_cells.append(sup_str)

        for task_key, _ in task_columns:
            task_scores = scores_dict.get(task_key, {})
            if m in task_scores:
                cell_str = _format_score_cell(
                    task_scores[m]["mean"],
                    task_scores[m]["std"],
                    red_threshold=red_threshold,
                    show_std=show_std,
                    bold=is_bold,
                )
            else:
                cell_str = "-"
            row_cells.append(cell_str)

        indent = "            " if span == 0 else "        "
        row_lines.append(indent + " & ".join(row_cells) + r" \\")

    lines = [
        r"\begin{table*}",
        r"    \centering",
        rf"    \begin{{tabular}}{{{col_spec}}}",
        r"    \toprule",
        header_line,
        r"    \midrule",
        *row_lines,
        r"    \bottomrule",
        r"    \end{tabular}",
        rf"    \caption{{{caption}}}",
        rf"    \label{{{label}}}",
        r"\end{table*}",
    ]
    return "\n".join(lines)


latex_table = render_cross_probe_latex_table(scores, enabled_models, show_std=False)
print(latex_table)

makedirs("saved", exist_ok=True)
Path("saved/st02_scores.json").write_text(json.dumps(scores, indent=2))
Path("saved/st02_table.tex").write_text(latex_table + "\n")

\begin{table*}
    \centering
    \begin{tabular}{l c c c c c}
    \toprule
        Model & \textit{d} & PE & $R^2_{\text{A} \to \text{A (micro)}}$ & $R^2_{\text{A} \to \text{B (micro)}}$ & $R^2_{\text{A} \to \text{B (texture)}}$ \\
    \midrule
        DINOv2 & S & Learned & \cellcolor{red!25} 0.80 & \cellcolor{red!25} 0.59 & \cellcolor{red!25} 0.49 \\
        \textbf{ALiBi-Dv2} & \textbf{S} & \textbf{ALiBI} & \cellcolor{green!25} \textbf{0.00} & \cellcolor{green!25} \textbf{-0.71} & \cellcolor{green!25} \textbf{-0.56} \\
    \bottomrule
    \end{tabular}
    \caption{$(x, y)$ linear probe $R^2$ scores trained on micrograph images (A) and evaluated on: held-out patches of the same image ($\text{A} \to \text{A}$, `micro'), patches of another micrograph image ($\text{A} \to \text{B}$, `micro'), and patches of a texture image ($\text{A} \to \text{B}$, `texture').}
    \label{tab:cross_image_linear_probe}
\end{table*}
